In the previous session we trained a model for predicting churn and evaluated it. Now let's deploy it

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [3]:
import os
import urllib.request

url = "https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv"
output_filename = "data_week3.csv"

#check if the file already exists
if not os.path.exists(output_filename):
    print(f"Downloading {output_filename}...")
    urllib.request.urlretrieve(url, output_filename)
    print("Download complete")
else:
    print(f"'{output_filename}' already exists. Skipping downloading")

'data_week3.csv' already exists. Skipping downloading


In [4]:
df = pd.read_csv('data_week3.csv')

df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.select_dtypes(include=['object', 'string']).columns)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

df.churn = (df.churn == 'yes').astype(int)

In [5]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)

In [6]:
numerical = ['tenure', 'monthlycharges', 'totalcharges']

categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod',
]

In [7]:
def train(df_train, y_train, C=1.0):
    dicts = df_train[categorical + numerical].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(dicts)

    model = LogisticRegression(C=C, max_iter=10000)
    model.fit(X_train, y_train)
    
    return dv, model

In [8]:
def predict(df, dv, model):
    dicts = df[categorical + numerical].to_dict(orient='records')

    X = dv.transform(dicts)
    y_pred = model.predict_proba(X)[:, 1]

    return y_pred

In [9]:
C = 1.0
n_splits = 5

In [10]:
kfold = KFold(n_splits=n_splits, shuffle=True, random_state=1)

scores = []

for train_idx, val_idx in kfold.split(df_full_train):
    df_train = df_full_train.iloc[train_idx]
    df_val = df_full_train.iloc[val_idx]

    y_train = df_train.churn.values
    y_val = df_val.churn.values

    dv, model = train(df_train, y_train, C=C)
    y_pred = predict(df_val, dv, model)

    auc = roc_auc_score(y_val, y_pred)
    scores.append(auc)

print('C=%s %.3f +- %.3f' % (C, np.mean(scores), np.std(scores)))

C=1.0 0.842 +- 0.007


In [11]:
scores

[0.8443375120691413,
 0.8449977651966627,
 0.8333012241621514,
 0.8347768913474451,
 0.8518166985660947]

In [12]:
dv, model = train(df_full_train, df_full_train.churn.values, C=1.0)
y_pred = predict(df_test, dv, model)

y_test = df_test.churn.values
auc = roc_auc_score(y_test, y_pred)
auc

0.8584032088573997

### Save the model

In [13]:
import pickle

In [14]:
output_file = f'model_C={C}.bin'
output_file

'model_C=1.0.bin'

In [15]:
#f_out = open(output_file, 'wb') 
#pickle.dump((dv, model), f_out)
#f_out.close()

In [16]:

import os
import pickle

output_file = f'model_C={C}.bin'

if os.path.exists(output_file):
    print(f"{output_file} already exists. Skipping save.")
else:
    with open(output_file, 'wb') as f_out: 
        pickle.dump((dv, model), f_out)
    print(f"Saved model to {output_file}")

model_C=1.0.bin already exists. Skipping save.


In [17]:
%ls -lh *.bin

-rw-r--r-- 1 pc pc 2.5K Aug 20 11:16 'model_C=1.0.bin'


### Load the model

In [18]:
import pickle

In [19]:
input_file = 'model_C=1.0.bin'

In [20]:
with open(input_file, 'rb') as f_in: #change wb to rb for reading
    dv, model = pickle.load(f_in)

In [21]:
dv,model

(DictVectorizer(sparse=False), LogisticRegression(max_iter=10000))

In [22]:
customer = {
    'gender': 'female',
    'seniorcitizen': 0,
    'partner': 'yes',
    'dependents': 'no',
    'phoneservice': 'no',
    'multiplelines': 'no_phone_service',
    'internetservice': 'dsl',
    'onlinesecurity': 'no',
    'onlinebackup': 'yes',
    'deviceprotection': 'no',
    'techsupport': 'no',
    'streamingtv': 'no',
    'streamingmovies': 'no',
    'contract': 'month-to-month',
    'paperlessbilling': 'yes',
    'paymentmethod': 'electronic_check',
    'tenure': 1,
    'monthlycharges': 29.85,
    'totalcharges': 29.85
}

In [23]:
X = dv.transform([customer])

In [24]:
y_pred = model.predict_proba(X)[0, 1]

In [25]:
print('input:', customer)
print('output:', y_pred)

input: {'gender': 'female', 'seniorcitizen': 0, 'partner': 'yes', 'dependents': 'no', 'phoneservice': 'no', 'multiplelines': 'no_phone_service', 'internetservice': 'dsl', 'onlinesecurity': 'no', 'onlinebackup': 'yes', 'deviceprotection': 'no', 'techsupport': 'no', 'streamingtv': 'no', 'streamingmovies': 'no', 'contract': 'month-to-month', 'paperlessbilling': 'yes', 'paymentmethod': 'electronic_check', 'tenure': 1, 'monthlycharges': 29.85, 'totalcharges': 29.85}
output: 0.6278927307895446


This customer is probably going to churn


### Making requests

In [26]:
import requests

In [27]:
url = 'http://localhost:9696/predict'

In [28]:
customer_1 = {
   "gender":"female",
   "seniorcitizen": 0,
   "partner":"yes",
   "dependents":"no",
   "phoneservice":"no",
   "multiplelines":"no_phone_service",
   "internetservice":"dsl",
   "onlinesecurity":"no",
   "onlinebackup":"yes",
   "deviceprotection":"no",
   "techsupport":"no",
   "streamingtv":"no",
   "streamingmovies":"no",
   "contract":"month-to-month",
   "paperlessbilling":"yes",
   "paymentmethod":"electronic_check",
   "tenure": 24,
   "monthlycharges": 29.85,
   "totalcharges": (24 *29.85)
}

In [29]:
customer_2 = {
   "gender":"female",
   "seniorcitizen": 0,
   "partner":"yes",
   "dependents":"no",
   "phoneservice":"no",
   "multiplelines":"no_phone_service",
   "internetservice":"dsl",
   "onlinesecurity":"no",
   "onlinebackup":"yes",
   "deviceprotection":"no",
   "techsupport":"no",
   "streamingtv":"no",
   "streamingmovies":"no",
   "contract":"month-to-month",
   "paperlessbilling":"yes",
   "paymentmethod":"electronic_check",
   "tenure": 1,
   "monthlycharges": 29.85,
   "totalcharges": 29.85
}


In [30]:
response = requests.post(url, json=customer_1).json() #change between customer_1 and customer_2

ConnectionError: HTTPConnectionPool(host='localhost', port=9696): Max retries exceeded with url: /predict (Caused by NewConnectionError("HTTPConnection(host='localhost', port=9696): Failed to establish a new connection: [Errno 111] Connection refused"))

In [ ]:
response

{'churn': False, 'churn_probability': 0.3392936364732968}

In [ ]:
if response['churn'] == True:
    print('sending promo email to', 'asdx-123d')
else:
    print(f'Customer not churning')

Customer not churning


- waitress-serve --host=127.0.0.1 --port=9696 predict:app

- gunicorn --bind 127.0.0.1:9696  predict:app

- gh repo clone Gachaupatrick/zindi

For execution in terminal using waitress/gunicorn

Customer 1 is not churning while Customer 2 get a promotional email